# Inspect the least-squares experiments

Read-only analysis of an existing run: **no simulations, job submissions, or result overwrites**.
Select the `FL_EF21_ENV` kernel, then run the cells in order. The default run below is job **54712913**.

- **A–B:** actual sampled initialization; iterations, not evaluation budgets.
- **C–D:** prescribed initialization to isolate mechanisms; no acquisition-cost comparison.

The `.err` log containing seven passing tests is normal: Python's unittest runner writes its report to stderr.


In [ ]:
from pathlib import Path
import csv
import json
import numpy as np
import matplotlib.pyplot as plt

# Find experiments/ whether the notebook starts here or at a repository root.
candidates = [
    parent / suffix
    for parent in [Path.cwd(), *Path.cwd().parents]
    for suffix in (Path("."), Path("experiments"), Path("TSGD/experiments"))
]
EXPERIMENT_DIR = next(
    (p.resolve() for p in candidates if (p / "least_squares.py").is_file()), None
)
if EXPERIMENT_DIR is None:
    raise FileNotFoundError("Open this notebook from the experiments folder or its repository.")

# EDIT THIS LINE to select another job, or use an absolute path to downloaded results.
RUN_DIR = EXPERIMENT_DIR / "results" / "cluster" / "full_54712913"
# For the earlier local run: RUN_DIR = EXPERIMENT_DIR / "results"

available = sorted((EXPERIMENT_DIR / "results").rglob("settings_and_summary.json"))
print("Available result folders:")
for path in available:
    print(" ", path.parent.relative_to(EXPERIMENT_DIR))
print("\nSelected:", RUN_DIR)

for name in ("curves.csv", "settings_and_summary.json"):
    if not (RUN_DIR / name).is_file():
        raise FileNotFoundError(
            f"Missing {RUN_DIR / name}. Edit RUN_DIR above; no other run is selected silently."
        )


In [ ]:
with (RUN_DIR / "settings_and_summary.json").open(encoding="utf-8") as handle:
    settings = json.load(handle)
with (RUN_DIR / "curves.csv").open(newline="", encoding="utf-8") as handle:
    raw_rows = list(csv.DictReader(handle))

series = {}
for row in raw_rows:
    key = (row["panel"], row["scenario"])
    converted = {
        name: (int(value) if name == "iteration" else float(value))
        for name, value in row.items() if name not in ("panel", "scenario")
    }
    series.setdefault(key, []).append(converted)
for key, rows in series.items():
    rows.sort(key=lambda r: r["iteration"])
    if [r["iteration"] for r in rows] != list(range(settings["steps"] + 1)):
        raise ValueError(f"Missing or duplicate iteration records for {key}")
specs = {(s["panel"], s["name"]): s for s in settings["scenarios"]}
if set(specs) != set(series):
    raise ValueError("The CSV scenarios do not match the JSON metadata.")

print(json.dumps(
    {key: settings.get(key) for key in (
        "runs", "steps", "dimension", "eta", "seed", "python_version",
        "numpy_version", "matplotlib_version", "runtime_environment", "comparison"
    )}, indent=2
))
print(f"Loaded {len(series)} scenarios, {len(raw_rows)} iteration records.")


## Original four-panel figure

This is the PNG saved by the job, without changing its appearance. The plots below rebuild individual panels from the CSV so you can zoom in or show the exact predictions.


In [ ]:
figure_path = RUN_DIR / "least_squares.png"
if figure_path.is_file():
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.imshow(plt.imread(figure_path))
    ax.axis("off")
    plt.show()
else:
    print("Saved PNG not found; the CSV plots below still work.")


## Plot controls and helpers

Edit `SHOW_EXACT`, `SHOW_BANDS`, or `MAX_ITER`, then rerun the desired panel. Solid/dashed colored curves are Monte Carlo means; dotted curves are **exact expectations**, not convergence upper bounds. Bands are ±2 pointwise Monte Carlo standard errors, not simultaneous confidence bands.


In [ ]:
SHOW_EXACT = True
SHOW_BANDS = True
MAX_ITER = None  # e.g. 60 to inspect the transient; None shows every iteration.

def table(headers, rows):
    formatted = [[f"{v:.5g}" if isinstance(v, (float, np.floating)) else str(v)
                  for v in row] for row in rows]
    widths = [max(len(str(h)), *(len(row[i]) for row in formatted))
              for i, h in enumerate(headers)]
    print(" | ".join(str(h).ljust(w) for h, w in zip(headers, widths)))
    print("-+-".join("-" * w for w in widths))
    for row in formatted:
        print(" | ".join(v.ljust(w) for v, w in zip(row, widths)))

def column(key, field):
    return np.array([row[field] for row in series[key]])

def positive(values):
    # Do not replace tiny positive data with an artificial error floor.
    return np.where(values > 0, values, np.nan)

def draw(ax, key, metric, label, color, style="-", marker=None, offset=0):
    k = column(key, "iteration")
    mean, se = column(key, f"{metric}_mean"), column(key, f"{metric}_se")
    ax.plot(k, positive(mean), color=color, ls=style, lw=1.8, label=label,
            marker=marker, ms=5, markerfacecolor="none",
            markevery=(offset, max(1, settings["steps"] // 10)))
    if SHOW_BANDS:
        ax.fill_between(k, positive(mean - 2 * se), positive(mean + 2 * se),
                        color=color, alpha=.12)
    if SHOW_EXACT:
        ax.plot(k, positive(column(key, f"{metric}_exact")),
                color=color, ls=":", lw=1.1, alpha=.9)

def plot_panel(panel, metric="gradient", show_estimator=False):
    names = {
        "A": "A — Noisy gradients, accurate differences (sampled initialization)",
        "B": "B — Fresh gradients and false convergence (sampled initialization)",
        "C": "C — Label noise cancels (controlled exact initialization)",
        "D": "D — Initial quality and similarity (controlled initialization)",
    }
    ylabel = {
        "gradient": r"Expected squared true-gradient norm",
        "estimator": r"Expected squared estimator norm",
        "tracking": r"Expected squared tracking error",
    }
    if metric not in ylabel:
        raise ValueError("metric must be gradient, estimator, or tracking")
    fig, ax = plt.subplots(figsize=(10, 5))
    keys = [key for key in specs if key[0] == panel]
    if not keys:
        raise ValueError(f"No panel {panel}")
    colors = ["#222222", "#d55e00", "#0072b2", "#009e73", "#cc79a7", "#56b4e9"]
    for index, key in enumerate(keys):
        spec = specs[key]
        color, style, marker, offset = colors[index % len(colors)], "-", None, 0
        if panel == "C" and spec["method"] == "TSGD":
            color = "#0072b2"
            marker = ["o", "s", "^"][index % 3]
            offset = index * max(1, settings["steps"] // 30)
        elif panel == "C":
            color, style = colors[(index - 3) % len(colors)], "--"
        elif panel == "D":
            color = "#0072b2" if spec["kappa"] == .05 else "#d55e00"
            style = "-" if spec["initial_mse"] == .001 else "--"
        draw(ax, key, metric, key[1], color, style, marker, offset)
        if show_estimator and metric == "gradient" and spec["method"] == "TSGD":
            draw(ax, key, "estimator", "TSGD estimator squared norm", color, "--")
    ax.set_yscale("log")
    ax.set_xlim(0, settings["steps"] if MAX_ITER is None else MAX_ITER)
    ax.set_xlabel("Iteration k")
    ax.set_ylabel(ylabel[metric])
    ax.set_title(names[panel] + (f"\neta={settings['eta']}; dotted = exact expectations"
                               if SHOW_EXACT else f"\neta={settings['eta']}"))
    ax.grid(True, which="major", alpha=.2)
    ax.legend(fontsize=9)
    fig.tight_layout()
    plt.show()


## A — Accurate differences despite noisy gradients

TSGD uses actual initial batches of 1, 128, or 512 samples. Compare their initial MSE and eventual true-gradient errors. Accurate transport preserves poor initial information too: the single-sample TSGD curve can be worse than SGD. This panel does **not** establish a computational saving.


In [ ]:
table(["Method", "Initialization", "Batch", "Expected E0", "Observed E0"],
      [[s["name"], s["initialization"], s["batch"], s["expected_E0"],
        series[key][0]["tracking_mean"]]
       for key, s in specs.items() if key[0] == "A"])
plot_panel("A")


## B — SGD shines and TSGD falsely converges

Compare TSGD's **true-gradient squared norm** with its **estimator squared norm**. Their separation is the false-convergence signal. SGD and the TSGD estimator may have nearly overlapping decaying curves; these are different quantities, not duplicated data.


In [ ]:
plot_panel("B", show_estimator=True)
table(["Method", "Final true gradient", "Final estimator", "Final tracking error"],
      [[key[1], rows[-1]["gradient_mean"], rows[-1]["estimator_mean"],
        rows[-1]["tracking_mean"]]
       for key, rows in series.items() if key[0] == "B"])


## C — Additive noise cancels

All noise levels use the same prescribed initial estimate and coupled feature samples. TSGD's three curves should coincide up to numerical roundoff; the marker shapes identify the overlapping curves. Changing label noise does not change transport, but would change the cost/accuracy of sampled initialization, which this panel intentionally holds fixed.


In [ ]:
plot_panel("C")
keys_c = [key for key, s in specs.items() if key[0] == "C" and s["method"] == "TSGD"]
reference = column(keys_c[0], "gradient_mean")
table(["TSGD setting", "Max difference between saved mean curves"],
      [[key[1], float(np.max(np.abs(column(key, "gradient_mean") - reference)))]
       for key in keys_c])
print("Run-time cancellation diagnostic:", settings["max_cancellation_difference"])
print("This recorded diagnostic includes final x/g differences and differences between mean curves.")
print("Individual trajectory histories were not saved in the CSV.")


## D — Initialization quality × similarity

Color distinguishes small/large δ, and line style distinguishes good/bad prescribed initialization. The population curvature and stepsize are fixed. The controlled errors are independent of later oracle samples; they are not estimates obtained from batches with comparable acquisition costs.


In [ ]:
plot_panel("D")


## Final errors versus exact expectations

`Exact at K` is the exact finite-time expectation. `Limit` is the constant-stepsize asymptotic prediction; finite-time errors need not equal it. The standardized discrepancy `(MC − exact)/SE` is descriptive, **not** a collection of independent hypothesis tests: curves share randomness and iterations are correlated. Zero SE is reported as undefined.


In [ ]:
comparison_rows = []
for key, spec in specs.items():
    row = series[key][-1]
    mean, exact, se = (row["gradient_mean"], row["gradient_exact"], row["gradient_se"])
    z = (mean - exact) / se if se > 0 else float("nan")
    comparison_rows.append([key[0], key[1], mean, se, exact,
                            spec["limiting_gradient_exact"], z])
table(["Panel", "Scenario", "MC at K", "SE", "Exact at K", "Limit", "(MC-exact)/SE"],
      comparison_rows)


## Optional detailed view

Change the panel, metric (`gradient`, `estimator`, or `tracking`), and iteration range. This only redraws stored data; it does not change the simulation parameters.


In [ ]:
MAX_ITER = 80
plot_panel("D", metric="tracking")
MAX_ITER = None
